In [2]:
import os
import re
import pandas as pd
from typing import List, Tuple

In [22]:
base_dir = "../data/caco2/caco2"
scale = 2
cell_ids = [0]

In [23]:
hr_dir = os.path.join(base_dir, "hr_div_1")
lr_dir = os.path.join(base_dir, f"hr_div_{scale}")

hr_files = os.listdir(hr_dir)
lr_files = os.listdir(lr_dir)


In [24]:
for f in hr_files[:5]:
    print(f, "->", "_".join(f.split("_")[1:-1]))

for f in lr_files[:5]:
    print(f, "->", "_".join(f.split("_")[1:-1]))

tile_HighRes1024-5_292_5376_6016_6144_6784_CELL0.tif -> HighRes1024-5_292_5376_6016_6144_6784
tile_HighRes1024-8_24_256_896_4352_4992_CELL2.tif -> HighRes1024-8_24_256_896_4352_4992
tile_HighRes1024-7_194_4864_5504_8448_9088_CELL1.tif -> HighRes1024-7_194_4864_5504_8448_9088
tile_HighRes1024-10_97_1792_2432_1536_2176_CELL0.tif -> HighRes1024-10_97_1792_2432_1536_2176
tile_HighRes1024-2_51_768_1408_1280_1920_CELL2.tif -> HighRes1024-2_51_768_1408_1280_1920
tile_LowRes512-8_158_1536_1856_1408_1728_CELL2.tif -> LowRes512-8_158_1536_1856_1408_1728
tile_LowRes512-12_88_512_832_4224_4544_CELL0.tif -> LowRes512-12_88_512_832_4224_4544
tile_LowRes512-20_241_2304_2624_384_704_CELL2.tif -> LowRes512-20_241_2304_2624_384_704
tile_LowRes512-13_235_2176_2496_2048_2368_CELL0.tif -> LowRes512-13_235_2176_2496_2048_2368
tile_LowRes512-17_312_1920_2240_256_576_CELL0.tif -> LowRes512-17_312_1920_2240_256_576


In [ ]:
fn = hr_files[100]
print(fn)
parts = fn.split("-")
if len(parts) < 2:
    print("Invalid filename format")
res_part = parts[0]  # e.g., tile_HighRes1024
rest = parts[1]
rest_core = rest.split("_")
print("rest_core", rest_core)
tile_idx = rest_core[0]
patch_size = rest_core[3]
key = "_".join(rest_core[1:-4])
cell = rest_core[-1].replace("CELL", "").replace(".tif", "")
full_key = f"{key}"
print("full_key", full_key)
print("cell", cell)
print("tile_idx", tile_idx)
print("patch_size", patch_size)


tile_HighRes1024-19_365_7936_8576_5632_6272_CELL1.tif
rest_core ['19', '365', '7936', '8576', '5632', '6272', 'CELL1.tif']
full_key 365_7936
cell 1
tile_idx 19
patch_size 8576
resolution 5632


In [6]:
if cell_ids is not None:
    cell_ids = set(map(str, cell_ids))
    hr_files = [f for f in hr_files if re.search(
            r'_CELL(\d+)', f) and re.search(r'_CELL(\d+)', f).group(1) in cell_ids]
    lr_files = [f for f in lr_files if re.search(
            r'_CELL(\d+)', f) and re.search(r'_CELL(\d+)', f).group(1) in cell_ids]

In [6]:
def extract_key(filename):
    # remove "tile_HighRes1024" or "tile_LowRes512" and CELL
    f = "_".join(filename.split("_")[:-1])
    return f.rsplit("-")[1:]

In [7]:
def extract_key(filename):
    return "_".join(filename.split("_")[1:-1])  # remove "tile_HighRes1024" or "tile_LowRes512" and CELL

hr_map = {
        (extract_key(f), re.search(r'_CELL(\d+)', f).group(1)): f
        for f in hr_files if "HighRes" in f
    }

dict(list(hr_map.items())[:2])

{('HighRes1024-5_292_5376_6016_6144_6784',
  '0'): 'tile_HighRes1024-5_292_5376_6016_6144_6784_CELL0.tif',
 ('HighRes1024-13_214_4096_4736_512_1152',
  '0'): 'tile_HighRes1024-13_214_4096_4736_512_1152_CELL0.tif'}

In [11]:
hr_map.keys()

dict_keys([('HighRes1024-5_292_5376_6016_6144_6784', '0'), ('HighRes1024-5_187_4096_4736_2048_2688', '2'), ('HighRes1024-21_145_1792_2432_6912_7552', '1'), ('HighRes1024-20_426_7424_8064_4864_5504', '2'), ('HighRes1024-11_229_5632_6272_2048_2688', '1'), ('HighRes1024-19_26_512_1152_5632_6272', '1'), ('HighRes1024-4_155_6656_7296_5632_6272', '1'), ('HighRes1024-13_214_4096_4736_512_1152', '0'), ('HighRes1024-9_128_4352_4992_6656_7296', '0'), ('HighRes1024-15_40_768_1408_5376_6016', '2'), ('HighRes1024-21_502_6400_7040_5632_6272', '2'), ('HighRes1024-18_567_7936_8576_6144_6784', '2'), ('HighRes1024-13_49_768_1408_1280_1920', '0'), ('HighRes1024-20_222_4096_4736_5120_5760', '1'), ('HighRes1024-15_292_4096_4736_5376_6016', '1'), ('HighRes1024-16_182_2304_2944_0_640', '2'), ('HighRes1024-14_29_768_1408_512_1152', '0'), ('HighRes1024-1_23_512_1152_8192_8832', '1'), ('HighRes1024-12_358_4864_5504_5632_6272', '1'), ('HighRes1024-15_326_4608_5248_4352_4992', '2'), ('HighRes1024-9_173_6400_7040_

In [8]:
GG = re.search(r'_CELL(\d+)', lr_files[1]).group(1)
lr_key = extract_key(lr_files[1])
hr_map.get((lr_key, GG), None)

In [9]:
matched_pairs = []
for lr_file in lr_files:
    if "LowRes" not in lr_file:
        print("no lr_file")
        continue

    match = re.search(r'_CELL(\d+)', lr_file)
    if not match:
        continue
    cell_id = match.group(1)
    lr_key = extract_key(lr_file)

    # Match HR file with same key and cell ID
    hr_filename = hr_map.get((lr_key, cell_id), None)
    if hr_filename:
        matched_pairs.append(
                (os.path.join(lr_dir, lr_file), os.path.join(hr_dir, hr_filename))
            )

len(matched_pairs)

0

In [6]:
def build_caco2_patch_dataframe(base_dir):
    """
    Build a dataframe with matched HR-LR image keys from Caco-2 dataset.

    Args:
        base_dir (str): Path to the folder containing hr_div_1, hr_div_2, hr_div_4, hr_div_8

    Returns:
        pd.DataFrame: Dataframe with columns: key, 1024, 512, 256, 128, slice, cell
    """

    div_folders = {
        '1024': 'hr_div_1',
        '512': 'hr_div_2',
        '256': 'hr_div_4',
        '128': 'hr_div_8'
    }

    records = {}

    for res, folder in div_folders.items():
        folder_path = os.path.join(base_dir, folder)
        if not os.path.exists(folder_path):
            continue
        for filename in os.listdir(folder_path):
            # print("filename", filename)
            if not filename.endswith(".tif"):
                continue
            parts = filename.split("-")
            if len(parts) < 2:
                continue
            res_part = parts[0]  # e.g., tile_HighRes1024
            rest = parts[1]
            rest_core = rest.split("_")

            if len(rest_core) < 2:
                continue

            tile_idx = rest_core[0]
            key = "_".join(rest_core[0:-5])
            cell = rest_core[-1].replace("CELL", "").replace(".tif", "")
            full_key = f"{key}"

            if full_key not in records:
                records[full_key] = {
                    'key': key,
                    'tile': int(tile_idx),
                    'cell': int(cell),
                    '1024': None,
                    '512': None,
                    '256': None,
                    '128': None
                }
            records[full_key][res] = os.path.join(filename)

    df = pd.DataFrame.from_dict(records, orient='index')
    df.reset_index(drop=True, inplace=True)
    return df

In [7]:
df = build_caco2_patch_dataframe("../data/caco2/caco2")

In [8]:
df.head()

,key,tile,cell,1024,512,256,128
0,5_292,5,0,tile_HighRes1024-5_292_5376_6016_6144_6784_CEL...,tile_LowRes512-5_292_2688_3008_3072_3392_CELL0...,tile_LowRes256-5_292_1344_1504_1536_1696_CELL2...,tile_LowRes128-5_292_672_752_768_848_CELL1.tif
1,8_24,8,2,tile_HighRes1024-8_24_256_896_4352_4992_CELL0.tif,tile_LowRes512-8_24_128_448_2176_2496_CELL0.tif,tile_LowRes256-8_24_64_224_1088_1248_CELL1.tif,tile_LowRes128-8_24_32_112_544_624_CELL1.tif
2,7_194,7,1,tile_HighRes1024-7_194_4864_5504_8448_9088_CEL...,tile_LowRes512-7_194_2432_2752_4224_4544_CELL1...,tile_LowRes256-7_194_1216_1376_2112_2272_CELL1...,tile_LowRes128-7_194_608_688_1056_1136_CELL2.tif
3,10_97,10,0,tile_HighRes1024-10_97_1792_2432_1536_2176_CEL...,tile_LowRes512-10_97_896_1216_768_1088_CELL2.tif,tile_LowRes256-10_97_448_608_384_544_CELL0.tif,tile_LowRes128-10_97_224_304_192_272_CELL1.tif
4,2_51,2,2,tile_HighRes1024-2_51_768_1408_1280_1920_CELL1...,tile_LowRes512-2_51_384_704_640_960_CELL2.tif,tile_LowRes256-2_51_192_352_320_480_CELL2.tif,tile_LowRes128-2_51_96_176_160_240_CELL0.tif


In [9]:
df.shape

(9937, 7)

In [10]:
df.to_csv("../data/caco2/caco2/caco2_patches.csv", index=False)